[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/glass_cyclide/glass_cyclide.ipynb)

# A Glass Dupin Cyclide

A ring of glass stands on a checkered floor, its tube thick on one side and thin on the other, with a round bite taken out of its thick side and a red ball behind it, while a camera circles it. The ring is a Dupin cyclide, a torus inverted in a sphere, and the light bends through it, reflects off it and fades inside it.

In the conformal model of space a point is held by its null vector, `up(position)`, and a quadric maps a point to its polar sphere, `Sphere <- Point`; its pairing with a point, `point & quadric(point)`, is zero on the surface and negative inside, and at a point of the surface its sphere is the tangent sphere. In this model the torus, a quartic surface in coordinates, is a quadric, and a sphere inversion is a versor, so `inversion >> torus(inversion << Point)` is the cyclide. A ball is a quadric as well, and the bite is the ring intersected with a ball's outside, the ball's quadric negated. A ray meets a solid where the largest of its quadrics' pairings changes sign; the shader, written in numga, finds that change by stepping and halving, and follows each pixel's light through the glass.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import stack
from examples.animation import save_figure, show_animation
from examples.quadrics.glass_cyclide import render
from examples.quadrics.glass_cyclide.core import (
    Euclidean, Point, Quadric, Rotation, Sphere, ball, cyclide, ga, infinity, moved, mv, sphere, torus_points, translator,
)
from examples.wgsl import Shader, supersampled

## 1. A torus inverted

Inverting the torus in a sphere off its centre swells the tube on the far side and narrows it on the near one. The torus's own points, carried by the same inversion, lie on the cyclide: the inversion carries the surface with them. A point's pairing scales with the square of its weight, its pairing with the point at infinity, so the readout divides it out.

In [ ]:
torus_radius, torus_tube = 2.5, 0.9
inversion = (translator(mv.x * 8.0) >> sphere(0 * mv.x, 6.25)).normalized()             # [] Sphere
surface = cyclide(torus_radius, torus_tube, inversion)                                   # [] Quadric
inverted = inversion >> torus_points(torus_radius, torus_tube, 24, 12)                    # [across, around] Sphere
on_surface = (inverted.dual() & surface(inverted.dual())) / (infinity | inverted).squared()   # [across, around] Scalar

In [ ]:
print("the cyclide at the inverted torus's points:", np.abs(on_surface.to_array()).max())

## 2. The solids

The ring is stood up by a motor, its bounding centre carried to the origin, tilted about x, and lifted to rest on the floor; the bite is a ball moved with it. Each solid carries the share of light it lets through and its refractive index: the glass lets nearly all of it through, the ball and the floor none.

In [ ]:
place = translator(mv.z * 2.25) * (mv.yz * (-np.radians(55.0) / 2)).exp() * translator(-mv.x * 2.04)   # [] Motor
shapes = [
    # The glass cyclide, less the bite: outside the bite's ball.
    [moved(surface, place), -ball(place >> sphere(mv.x * 0.6 + mv.y * 1.2 + mv.z * 0.9, 0.9))],
    [ball(sphere(mv.x * 0.6 + mv.y * 3.4 + mv.z * 1.1, 1.1))],
    # The floor: everything below the plane z = 0.
    [ball(-mv.z)],
]
counts = np.array([len(shape) for shape in shapes])
ends = np.cumsum(counts)
quadrics = stack([quadric.cast(Quadric) for shape in shapes for quadric in shape])          # [quadrics] Quadric
tiles = 0.8
solids = dict(first=ends - counts, end=ends,
              centre=stack([mv.z * 2.25, mv.x * 0.6 + mv.y * 3.4 + mv.z * 1.1, 0 * mv.x]), radius=np.array([2.6, 1.12, 60.0]),
              colour=np.array([[0.9, 0.96, 0.97], [0.85, 0.22, 0.16], [0.88, 0.87, 0.85]]),
              transparency=np.array([0.96, 0.0, 0.0]), refraction=np.array([1.5, 1.0, 1.0]),
              across=stack([0 * mv.x, 0 * mv.x, mv.x * tiles]), along=stack([0 * mv.y, 0 * mv.y, mv.y * tiles]))

## 3. The shader

Each invocation follows one pixel's light. At glass it refracts by Snell's law, with the outward direction square to the surface read off the tangent sphere, and adds what the reflected ray glimpses, by Fresnel's share in Schlick's approximation; past the critical angle all of it reflects, and inside the glass it fades by the glass's tint per unit length. An opaque surface ends the path with its colour, lit by the lamp and shadowed by what lies towards it, glass shadowing less.

In [ ]:
source = supersampled(2) + """
// A solid: the intersection of its quadrics, `quadrics[first]` up to `quadrics[end]`, each negative
// inside it, within a bounding sphere about `centre`. Its colour, the tint glass takes on per unit
// length; the share of light it lets through, and its refractive index; two directions whose unit steps
// lay a checker over it.
struct Solid {
    first: u32,
    end: u32,
    centre: Euclidean,
    radius: f32,
    colour: vec3<f32>,
    transparency: f32,
    refraction: f32,
    across: Euclidean,
    along: Euclidean,
}

var<storage, read> solids: array<Solid>;
var<storage, read> quadrics: array<Quadric>;
var<uniform> turn: Rotation;
var<uniform> eye: Euclidean;
var<uniform> lamp: Euclidean;
var<uniform> half_view: f32;
alias Sample = vec3<f32>;

const origin = (mv.w + mv.e) / 2;
const infinity = mv.e - mv.w;
const far = 1e6;
// The step a ray takes off a surface before it looks again; the steps across a bounding sphere, and the
// halvings of the one that crosses.
const clearance = 1e-3;
const n_steps = 40u;
const n_halvings = 18u;
const n_bounces = 8u;
// The look: light where the lamp does not reach, the light glass lets into its shadow, the checker's
// darkening, and the sky from the horizon up.
const ambient = 0.3;
const glass_shadow = 0.75;
const checker = 0.18;
const horizon = vec3<f32>(0.86, 0.88, 0.92);
const zenith = vec3<f32>(0.36, 0.5, 0.72);
// The sun seen in a reflection: its brightness, and how tightly it gathers about the lamp's direction.
const sun = 6.0;
const sun_size = 2000.0;

fn up(position: Euclidean) -> Point {
    return (origin + position + (position | position) / 2 * infinity).dual();
}

// How far a position lies outside a solid: the largest of its quadrics' forms, negative inside them all.
fn outside(solid: Solid, position: Euclidean) -> f32 {
    let point = up(position);
    var largest = -far;
    for (var index = solid.first; index < solid.end; index++) {
        largest = max(largest, point & quadrics[index](point));
    }
    return largest;
}

// The first distance along a ray at which it crosses a solid's surface, or `far`.
fn crossing(solid: Solid, origin: Euclidean, direction: Euclidean) -> f32 {
    // The stretch of the ray inside the bounding sphere.
    let along = (solid.centre - origin) | direction;
    let aside = solid.centre - origin - direction * along;
    let half_chord_squared = solid.radius * solid.radius - (aside | aside);
    let start = max(along - sqrt(max(half_chord_squared, 0.0)), clearance);
    let end = along + sqrt(max(half_chord_squared, 0.0));
    if (half_chord_squared <= 0 || end <= start) {
        return far;
    }
    let started_outside = outside(solid, origin + direction * start) > 0;
    var before = start;
    for (var step = 1u; step <= n_steps; step++) {
        let after = start + (end - start) * f32(step) / f32(n_steps);
        if ((outside(solid, origin + direction * after) > 0) != started_outside) {
            var low = before;
            var high = after;
            for (var halving = 0u; halving < n_halvings; halving++) {
                let middle = (low + high) / 2;
                if ((outside(solid, origin + direction * middle) > 0) == started_outside) {
                    low = middle;
                } else {
                    high = middle;
                }
            }
            return high;
        }
        before = after;
    }
    return far;
}

struct Hit {
    distance: f32,
    solid: u32,
}

fn nearest(origin: Euclidean, direction: Euclidean) -> Hit {
    var hit = Hit(far, 0u);
    for (var index = 0u; index < arrayLength(&solids); index++) {
        let distance = crossing(solids[index], origin, direction);
        if (distance < hit.distance) {
            hit = Hit(distance, index);
        }
    }
    return hit;
}

// The outward unit direction square to a solid at a position on it, from the tangent sphere of the
// quadric whose surface the position is on: the one the position lies least inside.
fn normal(solid: Solid, position: Euclidean) -> Euclidean {
    let point = up(position);
    var surface = solid.first;
    for (var index = solid.first; index < solid.end; index++) {
        if ((point & quadrics[index](point)) > (point & quadrics[surface](point))) {
            surface = index;
        }
    }
    let tangent = quadrics[surface](point);
    let direction = tangent.cast(Euclidean) + (tangent | infinity) * position;
    return direction / (direction | direction).square_root();
}

fn sky(direction: Euclidean) -> vec3<f32> {
    return mix(horizon, zenith, clamp(mv.z | direction, 0.0, 1.0)) + sun * pow(max(lamp | direction, 0.0), sun_size);
}

// The light towards the lamp from a position: dimmed by each solid in the way, opaque ones fully.
fn sunlight(position: Euclidean) -> f32 {
    var light = 1.0;
    for (var index = 0u; index < arrayLength(&solids); index++) {
        if (crossing(solids[index], position, lamp) < far) {
            light *= solids[index].transparency * glass_shadow;
        }
    }
    return light;
}

// An opaque surface's colour at a position: lit by the lamp, shadowed, and checkered.
fn surface(solid: Solid, position: Euclidean, facing: Euclidean) -> vec3<f32> {
    let lit = max(facing | lamp, 0.0) * sunlight(position + facing * clearance);
    let parity = abs((floor(solid.across | position) + floor(solid.along | position)) % 2.0);
    return solid.colour * (1 - checker * parity) * (ambient + (1 - ambient) * lit);
}

// What a ray glimpses without passing through glass: the surface it meets, or the sky.
fn glimpse(origin: Euclidean, direction: Euclidean) -> vec3<f32> {
    let hit = nearest(origin, direction);
    if (hit.distance >= far) {
        return sky(direction);
    }
    let solid = solids[hit.solid];
    let position = origin + direction * hit.distance;
    let facing = normal(solid, position);
    return mix(surface(solid, position, facing), sky(direction), solid.transparency);
}

fn unit(direction: Euclidean) -> Euclidean {
    return direction / (direction | direction).square_root();
}

// The light along the ray through a place on the image: from the camera's own -z, turned by its rotor.
fn sample(place: vec2<f32>) -> Sample {
    let across = screen(place) * half_view;
    var direction = unit(turn >> (-mv.z + mv.x * across.x + mv.y * across.y));
    var position = eye;
    var colour = vec3<f32>(0.0);
    var throughput = vec3<f32>(1.0);
    for (var bounce = 0u; bounce < n_bounces; bounce++) {
        let hit = nearest(position, direction);
        if (hit.distance >= far) {
            colour += throughput * sky(direction);
            break;
        }
        let solid = solids[hit.solid];
        position = position + direction * hit.distance;
        let outward = normal(solid, position);
        let entering = (outward | direction) < 0;
        // Light that crossed glass to get here faded by its tint, per unit length.
        throughput *= select(pow(solid.colour, vec3<f32>(hit.distance)), vec3<f32>(1.0), entering || solid.transparency == 0);
        let facing = select(-outward, outward, entering);
        colour += throughput * (1 - solid.transparency) * surface(solid, position, facing);
        // At glass the ray refracts, by Snell's law; past the critical angle all of it reflects.
        let cosine = -(facing | direction);
        let ratio = select(solid.refraction, 1 / solid.refraction, entering);
        let sine_squared = ratio * ratio * (1 - cosine * cosine);
        let reflected = direction + facing * (2 * cosine);
        let refracted = direction * ratio + facing * (ratio * cosine - sqrt(max(1 - sine_squared, 0.0)));
        // Schlick's approximation to Fresnel's share of reflected light.
        let normal_reflectance = ((solid.refraction - 1) / (solid.refraction + 1)) * ((solid.refraction - 1) / (solid.refraction + 1));
        let reflectance = normal_reflectance + (1 - normal_reflectance) * pow(1 - cosine, 5.0);
        let total = sine_squared > 1;
        colour += throughput * solid.transparency * select(reflectance, 0.0, total) * glimpse(position + reflected * clearance, reflected);
        throughput *= solid.transparency * select(1 - reflectance, 1.0, total);
        direction = select(refracted, reflected, total);
        position = position + direction * clearance;
        if (max(throughput.x, max(throughput.y, throughput.z)) < 0.01) {
            break;
        }
    }
    return colour;
}
"""

## 4. The camera's view

A camera is a rotation, from one looking along its own -z with y up, and an eye; the image comes from `supersampled(2)`, a turned grid of two samples a side in each pixel, averaged; the shader says what one sample sees, `fn sample(place: vec2<f32>) -> Sample`.

In [ ]:
lamp = mv.x * 0.5 - mv.y * 0.6 + mv.z * 0.9
lamp = lamp / (lamp | lamp).square_root()                                                 # [] Euclidean
target, distance, elevation, half_view = mv.z * 1.9, 11.0, np.radians(20.0), 0.31


def cameras(azimuths):
    turns = (mv.xy * (-azimuths / 2)).exp() * (mv.yz * (-(np.pi / 2 - elevation) / 2)).exp()   # [cameras] Rotation
    return turns, target + (turns >> mv.z) * distance                                     # [cameras] Rotation, [cameras] Euclidean


def frames(turns, eyes, shape):
    for turn, eye in zip(turns, eyes):
        traced = tracer(shape, quadrics=quadrics, solids=solids, turn=turn, eye=eye, lamp=lamp, half_view=half_view)["image"]
        yield np.round(np.clip(traced, 0, 1) ** (1 / 1.2) * 255).astype(np.uint8)


tracer = Shader(source, ga, dict(Sphere=Sphere, Euclidean=Euclidean, Point=Point, Rotation=Rotation, Quadric=Quadric))
still, = frames(*cameras(np.zeros(1)), (450, 720))
save_figure(render.draw_frame(still), "glass_cyclide");

## 5. Around the ring

Circling the ring shows it edge on and face on: through the thin side the floor bends little, through the thick side it folds over, and the bite shows as a lens cut into the glass.

In [ ]:
show_animation(list(frames(*cameras(np.linspace(0, 2 * np.pi, 48, endpoint=False)), (330, 528))), "glass_cyclide", 80)

In [ ]:
# checks
# The torus's points, inverted, lie on the cyclide.
np.testing.assert_allclose(on_surface.kernel, 0, atol=1e-10)